# 04 – Grad-CAM + Evaluation + Cross-Generator Test

- Grad-CAM grids (correct REAL / correct AI / misclassified)
- Confusion matrices side-by-side
- Cross-generator accuracy (limitation analysis)



In [ ]:
# --- Colab setup (run first) ---
# Runtime → Change runtime type → T4 GPU
!pip -q install kagglehub grad-cam

from pathlib import Path
from google.colab import drive
import kagglehub

# Everything we produce goes to Drive so it survives Colab restarts
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/ai-image-detector")
MODELS = OUT / "models"; REPORT = OUT / "report"
MODELS.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)

# Downloads CIFAKE once (~100 MB); contains train/{REAL,FAKE} and test/{REAL,FAKE}
DATA_ROOT = Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))
print("Data:", DATA_ROOT, "| Outputs:", OUT)

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

CKPT = MODELS / "effnet_b0_best.pt"
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

assert DATA_ROOT.exists(), f"Dataset not found at {DATA_ROOT}"
assert CKPT.exists(), f"Checkpoint not found at {CKPT}. Run notebook 03 first or upload the .pt file."
print("Data and checkpoint OK")

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(0.2, inplace=True),
        nn.Linear(1280, 1),
    )
    return model

model = build_model().to(DEVICE)
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
model.eval()
print("Model loaded")

In [ ]:
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds = datasets.ImageFolder(DATA_ROOT / "test", transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

all_preds, all_targets, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_ai = 1 - y.numpy()
        all_probs.extend(probs)
        all_preds.extend((probs >= 0.5).astype(int))
        all_targets.extend(y_ai)

acc = accuracy_score(all_targets, all_preds)
cm = confusion_matrix(all_targets, all_preds)
print(f"CIFAKE test accuracy: {acc:.4f}")
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm, display_labels=["REAL", "AI"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("EfficientNet-B0 – CIFAKE test")
plt.tight_layout()
plt.savefig(REPORT / "cm_effnet.png", dpi=150)
plt.show()
print("Saved", REPORT / "cm_effnet.png")

In [ ]:
target_layers = [model.features[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

def make_cam(pil_img, target_class):
    rgb = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    tensor = eval_tf(pil_img).unsqueeze(0).to(DEVICE)
    tensor.requires_grad_(True)
    grayscale = cam(input_tensor=tensor, targets=[BinaryClassifierOutputTarget(target_class)])[0]
    overlay = show_cam_on_image(rgb, grayscale, use_rgb=True, image_weight=0.5)
    return overlay

targets_np = np.array(all_targets)
preds_np = np.array(all_preds)
correct_real = np.where((targets_np == 0) & (preds_np == 0))[0]
correct_ai   = np.where((targets_np == 1) & (preds_np == 1))[0]
wrong        = np.where(targets_np != preds_np)[0]

print(f"Correct REAL: {len(correct_real)} | Correct AI: {len(correct_ai)} | Misclassified: {len(wrong)}")

def sample_paths(indices, n=4):
    idxs = random.sample(list(indices), min(n, len(indices)))
    return [test_ds.samples[i][0] for i in idxs]

grids = {
    "correct_REAL": (sample_paths(correct_real), 0),
    "correct_AI":   (sample_paths(correct_ai), 1),
    "misclassified": (sample_paths(wrong), None),
}

for title, (paths, tgt) in grids.items():
    if not paths:
        print(f"Skip {title}: no samples")
        continue
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, path in enumerate(paths):
        pil = Image.open(path).convert("RGB")
        axes[0, i].imshow(pil.resize((IMG_SIZE, IMG_SIZE)))
        axes[0, i].axis("off")
        if tgt is None:
            t = eval_tf(pil).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                p = torch.sigmoid(model(t).squeeze()).item()
            tgt_i = 1 if p >= 0.5 else 0
        else:
            tgt_i = tgt
        overlay = make_cam(pil, tgt_i)
        axes[1, i].imshow(overlay)
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original", fontsize=10)
    axes[1, 0].set_ylabel("Grad-CAM", fontsize=10)
    fig.suptitle(title.replace("_", " "), fontsize=13)
    plt.tight_layout()
    out = REPORT / f"gradcam_{title}.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"Saved {out}")

## Cross-generator test (bonus)

Collect 50–100 AI images from a *different* generator (Midjourney / Flux / DALL·E) and 50–100 real photos.
Put them in `MyDrive/ai-image-detector/cross_gen/REAL` and `.../cross_gen/FAKE` (any size, JPG/PNG). **Never used for training.**

In [ ]:
import io

def to_cifake_format(img):
    """Same as the backend: squash to a 32x32 JPEG before the 224 resize."""
    small = img.convert("RGB").resize((32, 32), Image.BICUBIC)
    buf = io.BytesIO(); small.save(buf, format="JPEG", quality=75); buf.seek(0)
    return Image.open(buf).convert("RGB")

CROSS_ROOT = OUT / "cross_gen"

def eval_folder(tf):
    ds = datasets.ImageFolder(CROSS_ROOT, transform=tf)
    preds, targets = [], []
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=32, shuffle=False):
            probs = torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).cpu().numpy()
            preds.extend((probs >= 0.5).astype(int))
            targets.extend(1 - y.numpy())  # FAKE=0 → AI=1
    return accuracy_score(targets, preds)

if CROSS_ROOT.exists():
    cross_direct  = eval_folder(eval_tf)
    cross_matched = eval_folder(transforms.Compose([transforms.Lambda(to_cifake_format), eval_tf]))
    print(f"CIFAKE test accuracy:                    {acc:.4f}")
    print(f"Cross-generator (direct 224 resize):     {cross_direct:.4f}")
    print(f"Cross-generator (32px JPEG, as in app):  {cross_matched:.4f}")
    print("\n→ The gap vs CIFAKE is the key limitation slide for the viva.")
else:
    print(f"Cross-gen folder not found at {CROSS_ROOT}. Skip for now.")

## Baseline CNN vs EfficientNet-B0 (report table)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import pandas as pd

class BaselineCNN(nn.Module):  # same definition as notebook 02
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)

baseline = BaselineCNN().to(DEVICE)
baseline.load_state_dict(torch.load(MODELS / "baseline_cnn_best.pt", map_location=DEVICE))
baseline.eval()

base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
base_loader = DataLoader(datasets.ImageFolder(DATA_ROOT / "test", transform=base_tf), batch_size=256, num_workers=2)
b_probs, b_targets = [], []
with torch.no_grad():
    for x, y in base_loader:
        b_probs.extend(torch.sigmoid(baseline(x.to(DEVICE))).cpu().numpy())
        b_targets.extend(1 - y.numpy())
b_probs = np.array(b_probs); b_preds = (b_probs >= 0.5).astype(int)

def metrics(t, p, prob):
    pr, rc, f1, _ = precision_recall_fscore_support(t, p, average="binary")
    return {"Accuracy": accuracy_score(t, p), "Precision": pr, "Recall": rc, "F1": f1, "ROC-AUC": roc_auc_score(t, prob)}

table = pd.DataFrame({
    "Baseline CNN (scratch, 32px)": metrics(b_targets, b_preds, b_probs),
    "EfficientNet-B0 (transfer, 224px)": metrics(all_targets, all_preds, all_probs),
}).T.round(4)
table.to_csv(REPORT / "metrics_comparison.csv")
display(table)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, t, p) in zip(axes, [("Baseline CNN", b_targets, b_preds), ("EfficientNet-B0", all_targets, all_preds)]):
    ConfusionMatrixDisplay(confusion_matrix(t, p), display_labels=["REAL", "AI"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.savefig(REPORT / "cm_side_by_side.png", dpi=150); plt.show()

## Export example images for the web app
Copies 3 REAL + 3 AI CIFAKE test images to `MyDrive/ai-image-detector/examples/`.
Download them and put them in `frontend/public/examples/`.

In [ ]:
import shutil
EX = OUT / "examples"; EX.mkdir(exist_ok=True)
random.seed(SEED)
for name, idxs in [("real", correct_real), ("ai", correct_ai)]:
    for k, i in enumerate(random.sample(list(idxs), 3), start=1):
        shutil.copy(test_ds.samples[i][0], EX / f"{name}_{k}.jpg")
print(sorted(p.name for p in EX.iterdir()))

### Sanity check

If Grad-CAM heatmaps focus on **borders/corners** instead of content → shortcut learning (model relies on padding artifacts or upscaling edges). Mention this honestly in the report.